# Surface Water WQI Classification 

This notebook uses the labelled **surface_wqi.csv** dataset. It keeps the held-out test set untouched. Gaussian augmentation and SMOTE are applied only to training data. Hyperparameter tuning uses SMOTE inside each CV fold.

In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import json

In [14]:
# 1. LOAD DATA
df = pd.read_csv("surface_wqi.csv")
print("Original dataset:", df.shape)
print("Target classes:\n", df["WQI_Class"].value_counts())

Original dataset: (42, 22)
Target classes:
 WQI_Class
Medium       19
Poor         19
Good          2
Very Poor     2
Name: count, dtype: int64


In [15]:
# 2. REMOVE TARGET / LEAKY / NEAR-EMPTY COLUMNS
near_empty = [c for c in df.columns if df[c].isna().all() or df[c].isna().sum() >= len(df)-1]
print("Near-empty columns:", near_empty)

drop_cols = near_empty + ["WQI", "WQI_Class", "WQI_Pred"]
df2 = df.drop(columns=drop_cols, errors="ignore").copy()

# Surface water: keep Iron because the existing classification workflow does not flag it as near-perfectly correlated with WQI.

X = df2.select_dtypes(include=np.number).copy()

# Keep Year to reproduce the existing 15-feature experiment.
print("Features:", list(X.columns))
print("Feature shape:", X.shape)

Near-empty columns: ['Odour', 'Lead(ppm)', 'Phosphates(ppm)', 'Pesticide (µg/l)']
Features: ['Year', 'pH', 'Turbidity', 'Conductivity', 'Chloride (ppm)', 'Sulphates(ppm)', 'Iron (ppm)', 'COD(ppm)', 'BOD(ppm)', 'DO(ppm)', 'Ammonia(ppm)', 'Nitrate(ppm)', 'Fluorides(ppm)', 'Total Bacterial Count (cfu/ml)', 'Total Fungal Count (cfu/ml)']
Feature shape: (42, 15)


In [16]:
# 3. ENCODE THE WQI CLASS ONCE
le = LabelEncoder()
y = pd.Series(le.fit_transform(df["WQI_Class"]), name="WQI_Class_Encoded")

print("Class encoding:")
for i, name in enumerate(le.classes_):
    print(i, "=", name)

Class encoding:
0 = Good
1 = Medium
2 = Poor
3 = Very Poor


## Why the shape stays `(n_samples, n_features)`

If the data have 42 observations and 15 predictor columns, the shape is `(42, 15)`.

For `test_size=0.20`:

\[
N_{test}\approx0.20N,\qquad N_{train}=N-N_{test}
\]

For 42 observations this gives 9 test and 33 training observations.

The number **15 does not change** because `train_test_split` divides rows, not columns.

In [17]:
# 4. TRAIN / TEST SPLIT — BEFORE AUGMENTATION
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)
print("Train classes:\n", y_train.value_counts().sort_index())
print("Test classes:\n", y_test.value_counts().sort_index())

Train: (33, 15)
Test : (9, 15)
Train classes:
 WQI_Class_Encoded
0     2
1    15
2    15
3     1
Name: count, dtype: int64
Test classes:
 WQI_Class_Encoded
1    4
2    4
3    1
Name: count, dtype: int64


In [18]:
# 5. GAUSSIAN AUGMENTATION — TRAINING DATA ONLY
def augment_data(X, y, copies=3, noise_fraction=0.01, seed=42):
    rng = np.random.RandomState(seed)
    std = X.std(skipna=True)
    X_parts = [X.copy()]
    y_parts = [y.copy()]

    for _ in range(copies):
        X_new = X.copy()
        for col in X.columns:
            if pd.isna(std[col]) or std[col] == 0:
                continue
            noise = rng.normal(0, noise_fraction * std[col], len(X))
            mask = X_new[col].notna()
            X_new.loc[mask, col] += noise[mask]

        X_parts.append(X_new)
        y_parts.append(y.copy())

    return pd.concat(X_parts, ignore_index=True), pd.concat(y_parts, ignore_index=True)

X_train_aug, y_train_aug = augment_data(X_train, y_train)

print("After Gaussian augmentation:", X_train_aug.shape)
print("Test remains untouched:", X_test.shape)

After Gaussian augmentation: (132, 15)
Test remains untouched: (9, 15)


In [19]:
# 6. FINAL TRAINING DATA — IMPUTE, SCALE, THEN SMOTE
imputer = KNNImputer(n_neighbors=5)
X_aug_imp = imputer.fit_transform(X_train_aug)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_aug_sc = scaler.fit_transform(X_aug_imp)
X_test_sc = scaler.transform(X_test_imp)

k_final = max(1, min(5, int(y_train_aug.value_counts().min()) - 1))
smote = SMOTE(random_state=42, k_neighbors=k_final)
X_train_sm, y_train_sm = smote.fit_resample(X_aug_sc, y_train_aug)

print("After imputation:", X_aug_imp.shape)
print("After scaling   :", X_aug_sc.shape)
print("After SMOTE     :", X_train_sm.shape)
print("SMOTE classes:\n", pd.Series(y_train_sm).value_counts().sort_index())
print("Test remains    :", X_test_sc.shape)

After imputation: (132, 15)
After scaling   : (132, 15)
After SMOTE     : (240, 15)
SMOTE classes:
 WQI_Class_Encoded
0    60
1    60
2    60
3    60
Name: count, dtype: int64
Test remains    : (9, 15)


### Why Gaussian augmentation and SMOTE increase rows but not columns

Gaussian augmentation creates noisy copies:

\[
X' = X + \epsilon,\qquad \epsilon\sim N(0,(\alpha s)^2)
\]

With 3 copies:

\[
N_{aug}=N_{train}(1+3)=33\times4=132.
\]

SMOTE creates synthetic minority observations. It does **not** create new features, so the column count remains 15.

The test set is never augmented or SMOTE-balanced because it must remain a genuinely held-out evaluation set.

In [20]:
# 7. HYPERPARAMETER GRIDS
grids = {
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {"model__n_estimators":[50,100,200],
         "model__max_depth":[None,5,10],
         "model__min_samples_split":[2,5]}
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {"model__n_estimators":[50,100],
         "model__learning_rate":[0.05,0.1,0.2],
         "model__max_depth":[3,5]}
    ),
    "XGBoost": (
        XGBClassifier(eval_metric="mlogloss", random_state=42, verbosity=0),
        {"model__n_estimators":[50,100,200],
         "model__learning_rate":[0.05,0.1],
         "model__max_depth":[3,5]}
    ),
    "SVM": (
        SVC(probability=True, random_state=42),
        {"model__C":[1,10,100],
         "model__gamma":["scale","auto",0.1],
         "model__kernel":["rbf","linear"]}
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors":[3,5,7,9],
         "model__weights":["uniform","distance"],
         "model__metric":["euclidean","manhattan"]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"model__max_depth":[None,3,5,10],
         "model__min_samples_split":[2,5,10],
         "model__criterion":["gini","entropy"]}
    )
}

### Why these hyperparameters are not in the training-metrics cell

They are **candidate settings**, not results.

GridSearchCV tries combinations and selects the combination with the highest weighted F1 during cross-validation.

Examples:

\[
n_{estimators}=50,100,200
\]

means try 50, 100 and 200 trees.

\[
learning\_rate=0.05,0.1,0.2
\]

controls the contribution of each boosting step.

\[
max\_depth=3,5
\]

controls tree depth and therefore model complexity.

In [21]:
# 8. GRID SEARCH
# 4 folds are used because the smallest augmented class has only 4 observations
# in the surface-water training set. SMOTE is inside the CV pipeline.

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

best_models = {}
best_params = {}
rows = []

# SMOTE k must be <= number of minority training samples in each CV fold - 1.
k_cv = max(1, min(5, int(y_train_aug.value_counts().min() / 4) - 1))
print("CV SMOTE k_neighbors:", k_cv)

for name, (base_model, params) in grids.items():

    pipe = Pipeline([
        ("imputer", KNNImputer(n_neighbors=5)),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42, k_neighbors=k_cv)),
        ("model", base_model)
    ])

    grid = GridSearchCV(
        pipe, params, cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )
    grid.fit(X_train_aug, y_train_aug)

    best_params[name] = grid.best_params_
    print(name, "->", grid.best_params_)

    # Final model: fit selected base model on the complete augmented + SMOTE training set.
    final_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}
    final_model = base_model.set_params(**final_params)
    final_model.fit(X_train_sm, y_train_sm)
    best_models[name] = final_model

    rows.append([name, grid.best_score_, grid.best_params_])

print("Grid search finished.")

CV SMOTE k_neighbors: 1
Random Forest -> {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 50}
Gradient Boosting -> {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 50}
XGBoost -> {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 50}
SVM -> {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
KNN -> {'model__metric': 'euclidean', 'model__n_neighbors': 3, 'model__weights': 'distance'}
Decision Tree -> {'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_split': 2}
Grid search finished.


In [22]:
# FINAL RESULTS

rows = []

for name, model in best_models.items():

    # Training predictions
    train_pred = model.predict(X_train_sm)

    train_acc = accuracy_score(y_train_sm, train_pred)
    train_precision = precision_score(
        y_train_sm, train_pred, average="weighted", zero_division=0
    )
    train_recall = recall_score(
        y_train_sm, train_pred, average="weighted", zero_division=0
    )
    train_f1 = f1_score(
        y_train_sm, train_pred, average="weighted", zero_division=0
    )
    train_mcc = matthews_corrcoef(y_train_sm, train_pred)

    # Testing predictions
    test_pred = model.predict(X_test_sc)

    test_acc = accuracy_score(y_test, test_pred)
    test_precision = precision_score(
        y_test, test_pred, average="weighted", zero_division=0
    )
    test_recall = recall_score(
        y_test, test_pred, average="weighted", zero_division=0
    )
    test_f1 = f1_score(
        y_test, test_pred, average="weighted", zero_division=0
    )
    test_mcc = matthews_corrcoef(y_test, test_pred)

    rows.append([
        name,
        train_acc, train_precision, train_recall,
        train_f1, train_mcc,
        test_acc, test_precision, test_recall,
        test_f1, test_mcc
    ])

columns = [
    "Model",
    "Training Accuracy",
    "Training Precision",
    "Training Recall",
    "Training F1",
    "Training MCC",
    "Testing Accuracy",
    "Testing Precision",
    "Testing Recall",
    "Testing F1",
    "Testing MCC"
]

results = pd.DataFrame(rows, columns=columns)

print("\nFINAL RESULTS")
print(results.round(4).to_string(index=False))

results.to_csv("final_results.csv", index=False)
print("\nSaved: final_results.csv")


FINAL RESULTS
            Model  Training Accuracy  Training Precision  Training Recall  Training F1  Training MCC  Testing Accuracy  Testing Precision  Testing Recall  Testing F1  Testing MCC
    Random Forest                1.0                 1.0              1.0          1.0           1.0            0.7778             0.6889          0.7778      0.7284       0.6162
Gradient Boosting                1.0                 1.0              1.0          1.0           1.0            0.8889             0.9111          0.8889      0.8871       0.8300
          XGBoost                1.0                 1.0              1.0          1.0           1.0            0.7778             0.8519          0.7778      0.7630       0.6847
              SVM                1.0                 1.0              1.0          1.0           1.0            0.6667             0.5926          0.6667      0.6095       0.4330
              KNN                1.0                 1.0              1.0          1.0    

In [24]:
# SAVE FINAL RESULTS — SURFACE WATER

results.to_csv("SW_final_results.csv", index=False)

results[
    ["Model",
     "Training Accuracy",
     "Training Precision",
     "Training Recall",
     "Training F1",
     "Training MCC"]
].to_csv("SW_training_results.csv", index=False)

results[
    ["Model",
     "Testing Accuracy",
     "Testing Precision",
     "Testing Recall",
     "Testing F1",
     "Testing MCC"]
].to_csv("SW_testing_results.csv", index=False)

pd.DataFrame({
    "Encoded": range(len(le.classes_)),
    "WQI_Class": le.classes_
}).to_csv("SW_class_encoding.csv", index=False)

print("Surface Water result files saved successfully.")

Surface Water result files saved successfully.


## Important interpretation

Training scores can be 1.00 because the final models are evaluated on the same augmented + SMOTE-balanced data used to fit them.

The **held-out testing metrics** are the main generalisation results.

The CV score is useful for model selection, but because the original dataset is extremely small and Gaussian copies are derived from the same observations, it should be described as **training-set cross-validation**, not independent external validation.